# This file will find the closest mall (straight-line distance) for each properties as the extra features, store location & distance.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from geopy.distance import great_circle

In [2]:
# read the files record propertry and mall respectively
property_data_origin = pd.read_csv("../data/curated/merged_data/merged_data_with_facility.csv")
mall_data = pd.read_csv("../data/landing/external_data/mall_coordinates.csv") # change path later

In [3]:
mall_data

,coordinates
0,"[-37.812733, 144.966947]"
1,"[-37.837395, 144.996158]"
2,"[-37.868967, 144.980617]"
3,"[-37.828989, 144.84627]"
4,"[-38.064493171914, 145.43517539621]"
...,...
212,"[-36.121153, 146.881917]"
213,"[-37.686167, 144.56127]"
214,"[-36.139008, 146.892456]"
215,"[-37.888057, 144.607036]"


In [4]:
# only select the needed features for running faster
property_data =  property_data_origin[['name','coordinates']]
property_data.head()

,name,coordinates
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]"
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]"
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]"
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]"
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]"


In [5]:
# This function will accept a string to parse coordinate string into point object
def parse_coordinates(coord_str):
    parts = coord_str.strip('[]').split(',')
    return (float(parts[0]), float(parts[1]))

In [6]:
# parse coordinate string into point object for both property & mall data
closest_mall = []
mall_distance = []
property_coordinates = property_data['coordinates'].apply(parse_coordinates)
all_mall_coordinates = mall_data['coordinates'].apply(parse_coordinates)

# find the closest mall for each property
for property_coord in property_coordinates:
    # By default: no nearest mall and the distance is positive infinity.
    min_distance = float('inf')
    closest_mall_geo = None
    
    for i, mall_coord in enumerate(all_mall_coordinates):
        # check if the value of the coordinate point is valid
        if np.isnan(property_coord).any() or np.isnan(mall_coord).any():
            continue
        # claculate the distance between property and mall, update if it is smallest
        distance = great_circle(property_coord, mall_coord).kilometers
        if distance < min_distance:
            min_distance = distance
            closest_mall_geo = mall_data.loc[i, 'coordinates']
    
    # add feature to store closest mall for the property
    closest_mall.append(closest_mall_geo)
    mall_distance.append(min_distance)
property_data['closest_mall'] = closest_mall
property_data['mall_distance(KM)'] = mall_distance
property_data

/tmp/ipykernel_123565/2408325822.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  property_data['closest_mall'] = closest_mall
/tmp/ipykernel_123565/2408325822.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  property_data['mall_distance(KM)'] = mall_distance


,name,coordinates,closest_mall,mall_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]","[-38.1184718, 145.3213262]",3.453903
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]","[-38.0604189, 145.3394612]",2.866025
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]","[-38.0604189, 145.3394612]",5.332842
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]","[-38.1184718, 145.3213262]",2.210922
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]","[-38.1184718, 145.3213262]",3.956745
...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]","[-37.8016476, 144.8977057]",1.411290
8798,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]","[-37.828989, 144.84627]",2.396280
8799,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]","[-37.828989, 144.84627]",2.273966
8800,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]","[-37.8016476, 144.8977057]",2.108881


In [7]:
# store closest mall information to the merged data
property_data_origin['closest_mall'] = closest_mall
property_data_origin['mall_distance(KM)'] = mall_distance
# save as a CSV file
property_data_origin.to_csv("../data/curated/merged_data/merged_data_with_facility.csv", index=False)
merged_data_with_facility = pd.read_csv("../data/curated/merged_data/merged_data_with_facility.csv")
merged_data_with_facility

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,...,avg_income_growth_rate(%),2021_population,avg_pop_growth_rates(%),crime_rate(%),closest_school,school_distance(KM),closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,...,3.028540,15038.0,129.962525,0.122595,"[-38.10602, 145.37876]",1.898006,"[-38.045325, 145.347181]",6.726397,"[-38.1184718, 145.3213262]",3.453903
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,...,3.028540,10652.0,29.771333,0.122595,"[-38.08468, 145.3638]",0.705427,"[-38.045325, 145.347181]",4.216162,"[-38.0604189, 145.3394612]",2.866025
2,7 Mortdale Lane Clyde North VIC 3978,490.0,2,2,1.0,3978.0,"[-38.0961758, 145.3800644]",POINT (145.3800644 -38.0961758),212031556.0,Clyde North - South,...,3.028540,15038.0,129.962525,0.122595,"[-38.10602, 145.37876]",1.100561,"[-38.045325, 145.347181]",6.344909,"[-38.0604189, 145.3394612]",5.332842
3,54 Walhallow Drive Clyde North VIC 3978,540.0,4,2,1.0,3978.0,"[-38.1133324, 145.3457396]",POINT (145.3457396 -38.1133324),212031556.0,Clyde North - South,...,3.028540,15038.0,129.962525,0.122595,"[-38.11488, 145.33828]",0.674921,"[-38.113312, 145.280832]",5.678594,"[-38.1184718, 145.3213262]",2.210922
4,10 Sicily Road Clyde North VIC 3978,520.0,4,2,2.0,3978.0,"[-38.1295789, 145.3642993]",POINT (145.3642993 -38.1295789),212031303.0,Cranbourne South,...,2.233051,17641.0,14.250357,0.197962,"[-38.12955, 145.33886]",2.225124,"[-38.113312, 145.280832]",7.522231,"[-38.1184718, 145.3213262]",3.956745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,630.0,2,1,0.0,3013.0,"[-37.8131463, 144.8909053]",POINT (144.8909053 -37.8131463),213031352.0,Yarraville,...,3.965740,15651.0,0.003162,0.197223,"[-37.8137, 144.8899]",0.107655,"[-37.800508, 144.895155]",1.454065,"[-37.8016476, 144.8977057]",1.411290
8798,47 Mill Avenue Yarraville VIC 3013,730.0,4,3,2.0,3013.0,"[-37.8222822, 144.872198]",POINT (144.872198 -37.8222822),213031352.0,Yarraville,...,3.965740,15651.0,0.003162,0.197223,"[-37.82104, 144.87443]",0.239821,"[-37.797407, 144.887421]",3.072331,"[-37.828989, 144.84627]",2.396280
8799,12 Adeney Street Yarraville VIC 3013,450.0,3,1,2.0,3013.0,"[-37.8163817, 144.8666543]",POINT (144.8666543 -37.8163817),213031352.0,Yarraville,...,3.965740,15651.0,0.003162,0.197223,"[-37.8126, 144.87466]",0.819385,"[-37.797407, 144.887421]",2.789294,"[-37.828989, 144.84627]",2.273966
8800,229B Somerville Road Yarraville VIC 3013,300.0,1,1,0.0,3013.0,"[-37.8124289, 144.8779569]",POINT (144.8779569 -37.8124289),213031352.0,Yarraville,...,3.965740,15651.0,0.003162,0.197223,"[-37.8126, 144.87466]",0.290245,"[-37.797407, 144.887421]",1.865866,"[-37.8016476, 144.8977057]",2.108881
